In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,roc_auc_score,confusion_matrix,classification_report

### Load Model Input Data

In [2]:
df=pd.read_csv("../Model_Input_Data/Model_Input_Data.csv")
print(df.shape)
df.head()

(5410, 36)


,Provider,PotentialFraud,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital.1,Avg_IP_Number_of_Days_in_Hospital.2,Avg_IP_Number_of_Days_in_Hospital.3,...,IschemicHeart_Count,Osteoporosis_Count,RheumatoidArthritis_Count,Stroke_Count,Avg_PartA_Months,Avg_PartB_Months_Mode,Avg_IP_Reimbursement,Avg_IP_Deductible,Avg_OP_Reimbursement,Avg_OP_Deductible
0,PRV51001,0,5,5,19400.000000,1068.0,0.0,0.0,6.0,0.0,...,2,18,16,19,12,12,18047.916667,890.000000,2537.500000,474.916667
1,PRV51003,1,62,53,9241.935484,1068.0,3.0,3.0,9.0,1.0,...,18,89,85,108,12,12,6814.017094,822.632479,2490.598291,664.529915
2,PRV51004,0,0,0,0.000000,0.0,0.0,0.0,0.0,0.0,...,40,95,97,122,12,12,4596.739130,454.144928,2095.144928,600.869565
3,PRV51005,1,0,0,0.000000,0.0,0.0,0.0,0.0,0.0,...,148,353,368,456,12,12,3717.232323,398.698990,1798.808081,475.965657
4,PRV51007,0,3,3,6333.333333,1068.0,4.0,4.0,6.0,0.0,...,18,41,42,49,12,12,3109.655172,423.517241,1497.241379,430.689655


In [3]:
df.PotentialFraud.value_counts()

0    4904
1     506
Name: PotentialFraud, dtype: int64

Here The data was Imbalanced.

### Checking Correlation Coefficent

- build a correlation matrix.
- identify Higly correlated variables.

In [4]:
corr_matrix=df.iloc[:,1:].corr()
corr_matrix.head()

,PotentialFraud,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital.1,Avg_IP_Number_of_Days_in_Hospital.2,Avg_IP_Number_of_Days_in_Hospital.3,OP_Claim_Count,...,IschemicHeart_Count,Osteoporosis_Count,RheumatoidArthritis_Count,Stroke_Count,Avg_PartA_Months,Avg_PartB_Months_Mode,Avg_IP_Reimbursement,Avg_IP_Deductible,Avg_OP_Reimbursement,Avg_OP_Deductible
PotentialFraud,1.000000,0.525393,0.522256,0.307556,0.319855,0.156278,0.157271,0.332043,0.203341,0.335803,...,0.369433,0.390565,0.388635,0.389683,0.009770,0.008369,0.127398,0.119897,-0.015204,-0.031547
IP_Claim_Count,0.525393,1.000000,0.997492,0.327985,0.398766,0.223728,0.224221,0.418265,0.228004,0.208758,...,0.325412,0.383660,0.379031,0.382209,0.009087,0.007868,0.165848,0.183794,-0.023449,-0.028603
IP_Benf_Count,0.522256,0.997492,1.000000,0.336416,0.409007,0.229987,0.230475,0.428677,0.231378,0.208244,...,0.328001,0.387186,0.382511,0.386015,0.009265,0.008031,0.170824,0.189412,-0.023422,-0.028680
Avg_IP_InscClaimAmtReimbursed,0.307556,0.327985,0.336416,1.000000,0.839577,0.672207,0.671091,0.827822,0.578765,0.022452,...,0.057874,0.079036,0.077457,0.078331,0.005905,0.010255,0.478656,0.411210,-0.007450,-0.026045
Avg_IP_DeductibleAmtPaid,0.319855,0.398766,0.409007,0.839577,1.000000,0.661765,0.662139,0.976286,0.566851,0.028770,...,0.074164,0.099966,0.097892,0.099275,-0.000936,0.003250,0.396898,0.433707,-0.029665,-0.042322


In [5]:
cols = corr_matrix.columns
corr_matrix['check'] = corr_matrix.apply(lambda row : [c for c in cols if 0.8<=abs(row[c])<1 ])

In [6]:
corr_matrix[['check']]

,check
PotentialFraud,[]
IP_Claim_Count,[IP_Benf_Count]
IP_Benf_Count,[IP_Claim_Count]
Avg_IP_InscClaimAmtReimbursed,"[Avg_IP_DeductibleAmtPaid, Avg_IP_Number_of_Da..."
Avg_IP_DeductibleAmtPaid,"[Avg_IP_InscClaimAmtReimbursed, Avg_IP_Number_..."
Avg_IP_Number_of_Days_in_Hospital,[Avg_IP_Number_of_Days_in_Hospital.1]
Avg_IP_Number_of_Days_in_Hospital.1,[Avg_IP_Number_of_Days_in_Hospital]
Avg_IP_Number_of_Days_in_Hospital.2,"[Avg_IP_InscClaimAmtReimbursed, Avg_IP_Deducti..."
Avg_IP_Number_of_Days_in_Hospital.3,[]
OP_Claim_Count,"[OP_Benf_Count, Total_Beneficiaries, RenalDise..."


### Train, Test Split

In [7]:
X_var = df.drop(columns = ['Provider','PotentialFraud'] )
y_var = df['PotentialFraud']

x_train,x_test,y_train,y_test = train_test_split(X_var,y_var,test_size = 0.2, stratify = y_var, random_state = 42)


In [8]:
print(x_train.shape)
print(x_test.shape)

(4328, 34)
(1082, 34)


In [9]:
print(y_train.value_counts())

0    3923
1     405
Name: PotentialFraud, dtype: int64


In [10]:
print(y_test.value_counts())

0    981
1    101
Name: PotentialFraud, dtype: int64


### Scaling

In [11]:
scaler=MinMaxScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

### Loading Logistic Regression Model

In [78]:
lr_estimate = LogisticRegression(max_iter = 1000,
                                 penalty = 'l2',
                                 solver = 'lbfgs',
                                 C = 1.8,
                                class_weight={0:0.35,1:0.65},
                                random_state = 42)

lr_estimate.fit(x_train,y_train)
y_pred = lr_estimate.predict(x_test)
#acc_score = accuracy_score(y_test,y_pred)
print("*****Classification Rport********")
print(classification_report(y_test,y_pred))
print("**** Confusion Matrix ***********")
print(confusion_matrix(y_test,y_pred))




*****Classification Rport********
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       981
           1       0.65      0.47      0.54       101

    accuracy                           0.93      1082
   macro avg       0.80      0.72      0.75      1082
weighted avg       0.92      0.93      0.92      1082

**** Confusion Matrix ***********
[[956  25]
 [ 54  47]]


In [79]:
print(lr_estimate.n_iter_)

[25]


In [80]:
Train_score = accuracy_score(y_train,lr_estimate.predict(x_train))
Test_score = accuracy_score(y_test,y_pred)
print("Train_score: ", Train_score)
print("Test_score: ", Test_score)
print("over_fiting: ", (Train_score - Test_score)*100)

Train_score:  0.9251386321626618
Test_score:  0.9269870609981515
over_fiting:  -0.1848428835489746


In [18]:
# Coefficients and Odds Ratios
coefficients = lr_estimate.coef_[0]


# Display feature importance using coefficients and odds ratios
feature_importance = pd.DataFrame({
    'Feature': X_var.columns,
    'Coefficient': coefficients,
    'abs_Coefficients': abs(coefficients)
})

feature_importance = feature_importance.sort_values(by = 'abs_Coefficients', ascending = False)

In [20]:
feature_importance

,Feature,Coefficient,abs_Coefficients
8,OP_Claim_Count,24.303898,24.303898
0,IP_Claim_Count,23.124487,23.124487
4,Avg_IP_Number_of_Days_in_Hospital,-5.158707,5.158707
2,Avg_IP_InscClaimAmtReimbursed,4.481323,4.481323
3,Avg_IP_DeductibleAmtPaid,1.090712,1.090712
6,Avg_IP_Number_of_Days_in_Hospital.2,0.450433,0.450433
27,Stroke_Count,0.000000,0.000000
22,Depression_Count,0.000000,0.000000
23,Diabetes_Count,0.000000,0.000000
24,IschemicHeart_Count,0.000000,0.000000
